In [ ]:
!pip install torch torchvision torchmetrics opencv-python pandas numpy matplotlib Pillow natsort tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from torch.autograd import Variable
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import glob
import time
import random
import sys
from PIL import Image
from datetime import datetime
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if torch.cuda.is_available():
    print(f"GPUs available: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
torch.cuda.empty_cache()

In [ ]:
def conv(in_channels, out_channels, kernel_size, bias=False, stride=1):
    return nn.Conv2d(
        in_channels, out_channels, kernel_size,
        padding=(kernel_size // 2), bias=bias, stride=stride)

def conv_down(in_chn, out_chn, bias=False):
    layer = nn.Conv2d(in_chn, out_chn, kernel_size=4, stride=2, padding=1, bias=bias)
    return layer

def default_conv(in_channels, out_channels, kernel_size, stride=1, bias=True):
    return nn.Conv2d(
        in_channels, out_channels, kernel_size,
        padding=(kernel_size//2), stride=stride, bias=bias)

class ResBlock(nn.Module):
    def __init__(
        self, conv, n_feats, kernel_size,
        bias=True, bn=False, act=nn.PReLU(), res_scale=1):
        super(ResBlock, self).__init__()
        m = []
        for i in range(2):
            if i == 0:
                m.append(conv(n_feats, 64, kernel_size, bias=bias))
            else:
                m.append(conv(64, n_feats, kernel_size, bias=bias))
            if bn:
                m.append(nn.BatchNorm2d(n_feats))
            if i == 0:
                m.append(act)
        self.body = nn.Sequential(*m)
        self.res_scale = res_scale
    def forward(self, x):
        res = self.body(x).mul(self.res_scale)
        res += x
        return res

class CAB(nn.Module):
    def __init__(self, n_feat, kernel_size, reduction, bias, act):
        super(CAB, self).__init__()
        modules_body = []
        modules_body.append(conv(n_feat, n_feat, kernel_size, bias=bias))
        modules_body.append(act)
        modules_body.append(conv(n_feat, n_feat, kernel_size, bias=bias))
        self.CA = CALayer(n_feat, reduction, bias=bias)
        self.body = nn.Sequential(*modules_body)
    def forward(self, x):
        res = self.body(x)
        res = self.CA(res)
        res += x
        return res

class CALayer(nn.Module):
    def __init__(self, channel, reduction=16, bias=False):
        super(CALayer, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv_du = nn.Sequential(
            nn.Conv2d(channel, channel // reduction, 1, padding=0, bias=bias),
            nn.ReLU(inplace=True),
            nn.Conv2d(channel // reduction, channel, 1, padding=0, bias=bias),
            nn.Sigmoid()
        )
    def forward(self, x):
        y = self.avg_pool(x)
        y = self.conv_du(y)
        return x * y

class SAM(nn.Module):
    def __init__(self, n_feat, kernel_size, bias):
        super(SAM, self).__init__()
        self.conv1 = conv(n_feat, n_feat, kernel_size, bias=bias)
        self.conv2 = conv(n_feat, 3, kernel_size, bias=bias)
    def forward(self, x, x_img):
        x1 = self.conv1(x)
        img = self.conv2(x) + x_img
        x1 = x1 + x
        return x1, img

class mergeblock(nn.Module):
    def __init__(self, n_feat, kernel_size, bias, subspace_dim=16):
        super(mergeblock, self).__init__()
        self.conv_block = conv(n_feat * 2, n_feat, kernel_size, bias=bias)
        self.num_subspace = subspace_dim
        self.subnet = conv(n_feat * 2, self.num_subspace, kernel_size, bias=bias)
    def forward(self, x, bridge):
        out = torch.cat([x, bridge], 1)
        b_, c_, h_, w_ = bridge.shape
        sub = self.subnet(out)
        V_t = sub.view(b_, self.num_subspace, h_*w_)
        V_t = V_t / (1e-6 + torch.abs(V_t).sum(axis=2, keepdims=True))
        V = V_t.permute(0, 2, 1)
        with torch.amp.autocast('cuda', enabled=False):
            mat = torch.matmul(V_t.float(), V.float())
            mat_inv = torch.inverse(mat)
            project_mat = torch.matmul(mat_inv, V_t.float())
            bridge_ = bridge.view(b_, c_, h_*w_)
            project_feature = torch.matmul(project_mat, bridge_.permute(0, 2, 1).float())
            bridge = torch.matmul(V.float(), project_feature).permute(0, 2, 1).view(b_, c_, h_, w_)
        out = torch.cat([x, bridge], 1)
        out = self.conv_block(out)
        return out + x

class Encoder(nn.Module):
    def __init__(self, n_feat, kernel_size, reduction, act, bias, scale_unetfeats, csff, depth=5):
        super(Encoder, self).__init__()
        self.body = nn.ModuleList()
        self.depth = depth
        for i in range(depth-1):
            self.body.append(UNetConvBlock(in_size=n_feat+scale_unetfeats*i, out_size=n_feat+scale_unetfeats*(i+1), downsample=True, relu_slope=0.2, use_csff=csff, use_HIN=True))
        self.body.append(UNetConvBlock(in_size=n_feat+scale_unetfeats*(depth-1), out_size=n_feat+scale_unetfeats*(depth-1), downsample=False, relu_slope=0.2, use_csff=csff, use_HIN=True))
    def forward(self, x, encoder_outs=None, decoder_outs=None):
        res = []
        if encoder_outs is not None and decoder_outs is not None:
            for i, down in enumerate(self.body):
                if (i+1) < self.depth:
                    x, x_up = down(x, encoder_outs[i], decoder_outs[-i-1])
                    res.append(x_up)
                else:
                    x = down(x)
        else:
            for i, down in enumerate(self.body):
                if (i+1) < self.depth:
                    x, x_up = down(x)
                    res.append(x_up)
                else:
                    x = down(x)
        return res, x

class UNetConvBlock(nn.Module):
    def __init__(self, in_size, out_size, downsample, relu_slope, use_csff=False, use_HIN=False):
        super(UNetConvBlock, self).__init__()
        self.downsample = downsample
        self.identity = nn.Conv2d(in_size, out_size, 1, 1, 0)
        self.use_csff = use_csff
        self.conv_1 = nn.Conv2d(in_size, out_size, kernel_size=3, padding=1, bias=True)
        self.relu_1 = nn.LeakyReLU(relu_slope, inplace=False)
        self.conv_2 = nn.Conv2d(out_size, out_size, kernel_size=3, padding=1, bias=True)
        self.relu_2 = nn.LeakyReLU(relu_slope, inplace=False)
        if downsample and use_csff:
            self.csff_enc = nn.Conv2d(out_size, out_size, 3, 1, 1)
            self.csff_dec = nn.Conv2d(in_size, out_size, 3, 1, 1)
            self.phi = nn.Conv2d(out_size, out_size, 3, 1, 1)
            self.gamma = nn.Conv2d(out_size, out_size, 3, 1, 1)
        if use_HIN:
            self.norm = nn.InstanceNorm2d(out_size//2, affine=True)
        self.use_HIN = use_HIN
        if downsample:
            self.downsample = conv_down(out_size, out_size, bias=False)
    def forward(self, x, enc=None, dec=None):
        out = self.conv_1(x)
        if self.use_HIN:
            out_1, out_2 = torch.chunk(out, 2, dim=1)
            out = torch.cat([self.norm(out_1), out_2], dim=1)
        out = self.relu_1(out)
        out = self.relu_2(self.conv_2(out))
        out += self.identity(x)
        if enc is not None and dec is not None:
            assert self.use_csff
            skip_ = F.leaky_relu(self.csff_enc(enc) + self.csff_dec(dec), 0.1, inplace=True)
            out = out * F.sigmoid(self.phi(skip_)) + self.gamma(skip_) + out
        if self.downsample:
            out_down = self.downsample(out)
            return out_down, out
        else:
            return out

class UNetUpBlock(nn.Module):
    def __init__(self, in_size, out_size, relu_slope):
        super(UNetUpBlock, self).__init__()
        self.up = nn.ConvTranspose2d(in_size, out_size, kernel_size=2, stride=2, bias=True)
        self.conv_block = UNetConvBlock(out_size*2, out_size, False, relu_slope)
    def forward(self, x, bridge):
        up = self.up(x)
        out = torch.cat([up, bridge], 1)
        out = self.conv_block(out)
        return out

class Decoder(nn.Module):
    def __init__(self, n_feat, kernel_size, reduction, act, bias, scale_unetfeats, depth=5):
        super(Decoder, self).__init__()
        self.body = nn.ModuleList()
        self.skip_conv = nn.ModuleList()
        for i in range(depth-1):
            self.body.append(UNetUpBlock(in_size=n_feat+scale_unetfeats*(depth-i-1), out_size=n_feat+scale_unetfeats*(depth-i-2), relu_slope=0.2))
            self.skip_conv.append(nn.Conv2d(n_feat+scale_unetfeats*(depth-i-1), n_feat+scale_unetfeats*(depth-i-2), 3, 1, 1))
    def forward(self, x, bridges):
        res = []
        for i, up in enumerate(self.body):
            x = up(x, self.skip_conv[i](bridges[-i-1]))
            res.append(x)
        return res

class DownSample(nn.Module):
    def __init__(self, in_channels, s_factor):
        super(DownSample, self).__init__()
        self.down = nn.Sequential(nn.Upsample(scale_factor=0.5, mode='bilinear', align_corners=False),
                                  nn.Conv2d(in_channels, in_channels + s_factor, 1, stride=1, padding=0, bias=False))
    def forward(self, x):
        x = self.down(x)
        return x

class UpSample(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UpSample, self).__init__()
        self.up = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False), nn.Conv2d(in_channels, out_channels, 1, stride=1, padding=0, bias=False))
    def forward(self, x):
        x = self.up(x)
        return x

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )
    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)
    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=False):
        super(UNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear
        self.inc = (DoubleConv(n_channels, 64))
        self.down1 = (Down(64, 128))
        self.down2 = (Down(128, 256))
        self.down3 = (Down(256, 512))
        factor = 2 if bilinear else 1
        self.down4 = (Down(512, 1024 // factor))
        self.up1 = (Up(1024, 512 // factor, bilinear))
        self.up2 = (Up(512, 256 // factor, bilinear))
        self.up3 = (Up(256, 128 // factor, bilinear))
        self.up4 = (Up(128, 64, bilinear))
        self.outc = (OutConv(64, n_classes))
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits
    def use_checkpointing(self):
        self.inc = torch.utils.checkpoint(self.inc)
        self.down1 = torch.utils.checkpoint(self.down1)
        self.down2 = torch.utils.checkpoint(self.down2)
        self.down3 = torch.utils.checkpoint(self.down3)
        self.down4 = torch.utils.checkpoint(self.down4)
        self.up1 = torch.utils.checkpoint(self.up1)
        self.up2 = torch.utils.checkpoint(self.up2)
        self.up3 = torch.utils.checkpoint(self.up3)
        self.up4 = torch.utils.checkpoint(self.up4)
        self.outc = torch.utils.checkpoint(self.outc)

class DenseLayer(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DenseLayer, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=3 // 2)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        return torch.cat([x, self.relu(self.conv(x))], 1)

class RDB(nn.Module):
    def __init__(self, in_channels, growth_rate, num_layers):
        super(RDB, self).__init__()
        self.layers = nn.Sequential(*[DenseLayer(in_channels + growth_rate * i, growth_rate) for i in range(num_layers)])
        self.lff = nn.Conv2d(in_channels + growth_rate * num_layers, growth_rate, kernel_size=1)
    def forward(self, x):
        return x + self.lff(self.layers(x))

class RDN(nn.Module):
    def __init__(self, input_channel, output_channel=3, num_features=64, growth_rate=64, num_blocks=2, num_layers=1):
        super(RDN, self).__init__()
        self.G0 = num_features
        self.G = growth_rate
        self.D = num_blocks
        self.C = num_layers
        self.sfe1 = nn.Conv2d(input_channel, num_features, kernel_size=3, padding=3 // 2)
        self.sfe2 = nn.Conv2d(num_features, num_features, kernel_size=3, padding=3 // 2)
        self.rdbs = nn.ModuleList([RDB(self.G0, self.G, self.C)])
        for _ in range(self.D - 1):
            self.rdbs.append(RDB(self.G, self.G, self.C))
        self.gff = nn.Sequential(
            nn.Conv2d(self.G * self.D, self.G0, kernel_size=1),
            nn.Conv2d(self.G0, self.G0, kernel_size=3, padding=3 // 2)
        )
        self.output = nn.Conv2d(self.G0, output_channel, kernel_size=3, padding=3 // 2)
    def forward(self, x):
        input = x
        sfe1 = self.sfe1(x)
        sfe2 = self.sfe2(sfe1)
        x = sfe2
        local_features = []
        for i in range(self.D):
            x = self.rdbs[i](x)
            local_features.append(x)
        x = self.gff(torch.cat(local_features, 1)) + sfe1
        output = self.output(x)
        return output

In [ ]:
def get_mean_value(batch):
    batch_size = batch.shape[0]
    list_mean_sorted = []
    list_indices = []
    largest_index = []
    medium_index = []
    smallest_index = []
    largest_channel = []
    medium_channel = []
    smallest_channel = []
    for bs in range(batch_size):
        image = batch[bs, :, :, :]
        mean = torch.mean(image, (2, 1))
        mean_I_sorted, indices = torch.sort(mean)
        list_mean_sorted.append(mean_I_sorted)
        list_indices.append(indices)
        largest_index.append(indices[2])
        medium_index.append(indices[1])
        smallest_index.append(indices[0])
        largest_channel.append(torch.unsqueeze(image[indices[2], :, :], 0))
        medium_channel.append(torch.unsqueeze(image[indices[1], :, :], 0))
        smallest_channel.append(torch.unsqueeze(image[indices[0], :, :], 0))
    list_mean_sorted = torch.stack(list_mean_sorted)
    list_indices = torch.stack(list_indices)
    largest_index = torch.stack(largest_index)
    medium_index = torch.stack(medium_index)
    smallest_index = torch.stack(smallest_index)
    largest_channel = torch.stack(largest_channel)
    medium_channel = torch.stack(medium_channel)
    smallest_channel = torch.stack(smallest_channel)
    return list_mean_sorted, list_indices, largest_channel, medium_channel, smallest_channel, largest_index, medium_index, smallest_index

def mapping_index(batch, value, index):
    batch_size = batch.shape[0]
    new_batch = []
    for bs in range(batch_size):
        image = batch[bs, :, :, :]
        image[index[bs], :, :] = value[bs]
        new_batch.append(image)
    new_batch = torch.stack(new_batch)
    return new_batch

class BasicBlock(nn.Module):
    def __init__(self):
        super(BasicBlock, self).__init__()
        print('Loading subnetworks .....')
        Z_Net = [RDN(3)]
        self.Z_Net = nn.Sequential(*Z_Net)
        self.t_1D_Net = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=1, kernel_size=1, bias=False),
            nn.ReLU()
        )
        self.alpha = nn.Parameter(torch.tensor([3.001]), requires_grad=True)
        self.beta = nn.Parameter(torch.tensor([3.001]), requires_grad=True)
        self.eta = nn.Parameter(torch.tensor([3.001]), requires_grad=True)
        self.lambda_1 = nn.Parameter(torch.tensor([1.001]), requires_grad=True)
        self.lambda_2 = nn.Parameter(torch.tensor([1.001]), requires_grad=True)
    def forward(self, I, t_p, B_p, B, t, J, Y, Z, Q, R, u, v, w_1, w_2, eps=1e-6):
        alpha = self.alpha
        beta = self.beta
        eta = self.eta
        lambda_1 = self.lambda_1
        lambda_2 = self.lambda_2
        gamma_1 = 0.3
        gamma_2 = 0.7
        list_mean_sorted, list_indices, J_l, J_m, J_s, largest_index, medium_index, smallest_index = get_mean_value(J)
        J_l_bar = torch.unsqueeze(torch.unsqueeze(torch.unsqueeze(list_mean_sorted[:,2],1),1),1).to(DEVICE)
        J_m_bar = torch.unsqueeze(torch.unsqueeze(torch.unsqueeze(list_mean_sorted[:,1],1),1),1).to(DEVICE)
        J_s_bar = torch.unsqueeze(torch.unsqueeze(torch.unsqueeze(list_mean_sorted[:,0],1),1),1).to(DEVICE)
        J_m_bar = J_l - u + (1.0/lambda_1)*w_1
        J_s_bar = J_l - v + (1.0/lambda_2)*w_2
        J_l = J_l.to(DEVICE)
        J_m = J_m.to(DEVICE)
        J_s = J_s.to(DEVICE)
        J_m = J_m + torch.mul(J_l_bar - J_m_bar, J_l)
        J_s = J_s + torch.mul(J_l_bar - J_s_bar, J_l)
        J = mapping_index(J.clone(), J_m.clone(), medium_index)
        J = mapping_index(J.clone(), J_s.clone(), smallest_index)
        u = torch.sign((J_l_bar - J_m_bar + (1.0/lambda_1)*w_1)) * F.relu(torch.abs((J_l_bar - J_m_bar + (1.0/lambda_1)*w_1)) - (1.0/lambda_1), inplace=False)
        v = torch.sign((J_l_bar - J_s_bar + (1.0/lambda_2)*w_2)) * F.relu(torch.abs((J_l_bar - J_s_bar + (1.0/lambda_2)*w_2)) - (1.0/lambda_2), inplace=False)
        w_1 = w_1 + lambda_1*(J_l_bar - J_m_bar - u)
        w_2 = w_2 + lambda_2*(J_l_bar - J_s_bar - v)
        D = torch.ones(I.shape).to(DEVICE)
        B = (gamma_1*B_p - (J*t - I)*(1 - t))/((1.0 - t)*(1 - t) + gamma_1)
        B = torch.mean(B, (2,3), True)
        B = B*D
        t = (gamma_2*t_p + eta*Z - R - (B - I)*(J - B))/((J - B)*(J - B) + gamma_2 + eta)
        t = self.t_1D_Net(t)
        t = torch.cat((t, t, t), 1)
        J = (beta*Y - Q - (B*(1.0 - t) - I)*t)/(t*t + beta)
        Z = self.Z_Net(t + (1.0/eta) * R)
        Q = Q + beta*(J - Y)
        R = R + eta*(t - Z)
        return B, t, J, Y, Z, Q, R, u, v, w_1, w_2, beta

class IPMM(nn.Module):
    def __init__(self, in_c=3, out_c=3, n_feat=80, scale_unetfeats=48, scale_orsnetfeats=32, num_cab=8, kernel_size=3, reduction=4, bias=False):
        super(IPMM, self).__init__()
        act = nn.PReLU()
        self.shallow_feat2 = nn.Sequential(conv(3, n_feat, kernel_size, bias=bias), CAB(n_feat, kernel_size, reduction, bias=bias, act=act))
        self.stage2_encoder = Encoder(n_feat, kernel_size, reduction, act, bias, scale_unetfeats, depth=4, csff=True)
        self.stage2_decoder = Decoder(n_feat, kernel_size, reduction, act, bias, scale_unetfeats, depth=4)
        self.sam23 = SAM(n_feat, kernel_size=1, bias=bias)
        self.r1 = nn.Parameter(torch.Tensor([0.5]))
        self.concat12 = conv(n_feat * 2, n_feat, kernel_size, bias=bias)
        self.merge12 = mergeblock(n_feat, 3, True)
    def forward(self, x2_img, stage1_img, feat1, res1, x2_samfeats):
        x2 = self.shallow_feat2(x2_img)
        x2_cat = self.merge12(x2, x2_samfeats)
        feat2, feat_fin2 = self.stage2_encoder(x2_cat, feat1, res1)
        res2 = self.stage2_decoder(feat_fin2, feat2)
        x3_samfeats, stage2_img = self.sam23(res2[-1], x2_img)
        return x3_samfeats, stage2_img, feat2, res2

class BLUE_Net(torch.nn.Module):
    def __init__(self, LayerNo):
        super(BLUE_Net, self).__init__()
        self.LayerNo = LayerNo
        net_layers = []
        for i in range(LayerNo):
            net_layers.append(BasicBlock())
        self.uunet = nn.ModuleList(net_layers)
        in_c = 3
        out_c = 3
        n_feat = 40
        scale_unetfeats = 20
        scale_orsnetfeats = 16
        num_cab = 8
        kernel_size = 3
        reduction = 4
        bias = False
        act = nn.PReLU()
        self.shallow_feat1 = nn.Sequential(conv(3, n_feat, kernel_size, bias=bias), CAB(n_feat, kernel_size, reduction, bias=bias, act=act))
        self.stage1_encoder = Encoder(n_feat, kernel_size, reduction, act, bias, scale_unetfeats, depth=4, csff=True)
        self.stage1_decoder = Decoder(n_feat, kernel_size, reduction, act, bias, scale_unetfeats, depth=4)
        self.sam12 = SAM(n_feat, kernel_size=1, bias=bias)
        self.r1 = nn.Parameter(torch.Tensor([0.5]))
        self.concat12 = conv(n_feat * 2, n_feat, kernel_size, bias=bias)
        self.merge12 = mergeblock(n_feat, 3, True)
        self.basic = IPMM(in_c=3, out_c=3, n_feat=40, scale_unetfeats=20, scale_orsnetfeats=16, num_cab=8, kernel_size=3, reduction=4, bias=False)
    def forward(self, I, t_p, B_p):
        bs, _, _, _ = I.shape
        B = torch.zeros((bs, 3, 1, 1)).to(DEVICE)
        t = torch.zeros(I.shape).to(DEVICE)
        J = I.to(DEVICE)
        X = torch.zeros(I.shape).to(DEVICE)
        Y = torch.zeros(I.shape).to(DEVICE)
        Z = torch.zeros(I.shape).to(DEVICE)
        P = torch.zeros(I.shape).to(DEVICE)
        Q = torch.zeros(I.shape).to(DEVICE)
        R = torch.zeros(I.shape).to(DEVICE)
        u = torch.zeros((bs, 1, 1, 1)).to(DEVICE)
        v = torch.zeros((bs, 1, 1, 1)).to(DEVICE)
        w_1 = torch.zeros((bs, 1, 1, 1)).to(DEVICE)
        w_2 = torch.zeros((bs, 1, 1, 1)).to(DEVICE)
        list_output = []
        list_B = []
        list_t = []
        beta = torch.tensor([3.001]).to(DEVICE)
        x1_img = J + (1.0/beta) * Q
        x1 = self.shallow_feat1(x1_img)
        feat1, feat_fin1 = self.stage1_encoder(x1)
        res1 = self.stage1_decoder(feat_fin1, feat1)
        x2_samfeats, stage1_img = self.sam12(res1[-1], x1_img)
        Y = stage1_img
        for j in range(self.LayerNo):
            [B, t, J, Y, Z, Q, R, u, v, w_1, w_2, beta] = self.uunet[j](I, t_p, B_p, B, t, J, Y, Z, Q, R, u, v, w_1, w_2)
            img = J + (1.0/beta) * Q
            x2_samfeats, stage1_img, feat1, res1 = self.basic(img, stage1_img, feat1, res1, x2_samfeats)
            Y = stage1_img
            if j < self.LayerNo - 1:
                list_output.append(J.detach())
                list_B.append(B.detach())
                list_t.append(t.detach())
            else:
                list_output.append(J)
                list_B.append(B)
                list_t.append(t)
        return list_output, list_B, list_t

In [ ]:
class UIEBDataset(Dataset):
    def __init__(self, root, transform=None, is_test=False):
        if transform is not None:
            self.transform = transforms.Compose(transform)
        else:
            self.transform = transforms.Compose([
                transforms.Resize((256, 256)),
                transforms.ToTensor(),
            ])
        self.input_files, self.gt_files, self.t_p_files, self.B_p_files = self.get_file_paths(root, is_test)
        self.len = min(len(self.input_files), len(self.gt_files), len(self.t_p_files))
    def __getitem__(self, index):
        input_image = Image.open(self.input_files[index % self.len])
        gt_image = Image.open(self.gt_files[index % self.len])
        t_p = Image.open(self.t_p_files[index % self.len])
        B_p = Image.open(self.B_p_files[index % self.len])
        input_image = self.transform(input_image)
        gt_image = self.transform(gt_image)
        t_p = self.transform(t_p)
        B_p = self.transform(B_p)
        return {"inp": input_image, "gt": gt_image, "t": t_p, "B": B_p}
    def __len__(self):
        return self.len
    def get_file_paths(self, root, is_test):
        if is_test:
            input_files = sorted(glob.glob(os.path.join(root, 'input') + "/*.*"))
            t_p_files = sorted(glob.glob(os.path.join(root, 't_prior') + "/*.*"))
            B_p_files = sorted(glob.glob(os.path.join(root, 'b_prior') + "/*.*"))
            gt_files = []
        else:
            input_files = sorted(glob.glob(os.path.join(root, 'input') + "/*.*"))
            gt_files = sorted(glob.glob(os.path.join(root, 'reference-890') + "/*.*"))
            t_p_files = sorted(glob.glob(os.path.join(root, 't_prior') + "/*.*"))
            B_p_files = sorted(glob.glob(os.path.join(root, 'b_prior') + "/*.*"))
        return input_files, gt_files, t_p_files, B_p_files

In [ ]:
try:
    from torchmetrics.image import PeakSignalNoiseRatio
    psnr_metric = PeakSignalNoiseRatio(data_range=1.0)
except ImportError:
    class PeakSignalNoiseRatio(nn.Module):
        def __init__(self, data_range=1.0):
            super().__init__()
            self.data_range = data_range
        def forward(self, preds, target):
            mse = torch.mean((preds - target) ** 2)
            if mse == 0:
                return torch.tensor(100.0)
            return 10 * torch.log10(self.data_range**2 / mse)
    psnr_metric = PeakSignalNoiseRatio(data_range=1.0)

def set_seed(seed, torch_set_seed=True):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    if torch_set_seed:
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def validate_model(model, dataloader, loss_func, args):
    model.eval()
    val_loss = 0.0
    psnr_val = 0.0
    psnr_fn = psnr_metric.to(DEVICE)
    with torch.inference_mode():
        for batch in tqdm(dataloader, desc="Validating", leave=False):
            imgs_distorted = batch["inp"].to(DEVICE, non_blocking=True)
            imgs_good_gt   = batch["gt"].to(DEVICE, non_blocking=True)
            t_p            = batch["t"].to(DEVICE, non_blocking=True)
            B_p            = batch["B"].to(DEVICE, non_blocking=True)
            if t_p.shape[1] == 1:
                t_p = t_p.repeat(1, 3, 1, 1)
            list_J, _, _ = model(imgs_distorted, t_p, B_p)
            output = list_J[-1]
            val_loss += loss_func(output, imgs_good_gt).item()
            psnr_val += psnr_fn(output, imgs_good_gt).item()
    n = len(dataloader.dataset)
    return (val_loss / n) if n > 0 else 0, (psnr_val / n) if n > 0 else 0

def training(args):
    transform = [transforms.Resize((256, 256)), transforms.ToTensor()]
    print("Training set loading...")
    import platform
    effective_workers = 0 if platform.system() == 'Windows' else args.num_workers

    train_dataloader = DataLoader(
        UIEBDataset(args.train_dir, transform=transform),
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=effective_workers,
        pin_memory=True,
        persistent_workers=True
    )

    print('Validation set loading...')
    try:
        val_dataset = UIEBDataset(args.val_dir, transform=transform)
        val_dataloader = DataLoader(val_dataset, batch_size=1, shuffle=True, num_workers=1, pin_memory=True, persistent_workers=True)
    except FileNotFoundError:
        print("Warning: Validation directory is empty or invalid. Skipping validation.")
        val_dataloader = None

    loss_1 = torch.nn.L1Loss()
    
    model = BLUE_Net(LayerNo=args.n_layers)
    model.to(DEVICE)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, betas=(0.9, 0.99))
    scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

    save_path = os.path.join(args.train_logs, 'save_path')
    os.makedirs(save_path, exist_ok=True)
    metrics_path = os.path.join(save_path, 'training_metrics.csv')

    train_losses = []
    val_losses = []
    val_psnrs = []
    epoch_0 = 1
    best_acc = 0.0

    if args.resume and os.path.exists(args.resume):
        print(f'Resuming from {args.resume}')
        ckpt = torch.load(args.resume, map_location=DEVICE)
        state_dict = {k.replace('module.', ''): v for k, v in ckpt['model_state_dict'].items()}
        model.load_state_dict(state_dict)
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        epoch_0 = ckpt['epoch'] + 1
        best_acc = ckpt.get('best_acc', 0.0)

        if os.path.exists(metrics_path):
            df_prev = pd.read_csv(metrics_path)
            train_losses = df_prev['train_loss'].tolist()
            val_losses = df_prev['val_loss'].tolist()
            val_psnrs = df_prev['val_psnr'].tolist()

    for epoch in range(epoch_0, args.epochs + 1):
        model.train()
        epoch_loss = 0.0

        pbar = tqdm(train_dataloader, desc=f"Epoch {epoch}/{args.epochs}", leave=True)

        for i, batch in enumerate(pbar):
            imgs_distorted = batch["inp"].to(DEVICE, non_blocking=True)
            imgs_good_gt   = batch["gt"].to(DEVICE, non_blocking=True)
            t_p            = batch["t"].to(DEVICE, non_blocking=True)
            B_p            = batch["B"].to(DEVICE, non_blocking=True)

            if t_p.shape[1] == 1:
                t_p = t_p.repeat(1, 3, 1, 1)

            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                output_J, _, _ = model(imgs_distorted, t_p, B_p)
                output = output_J[-1]
                total_loss = loss_1(output, imgs_good_gt)

            scaler.scale(total_loss).backward()
            scaler.step(optimizer)
            scaler.update()

            loss_item = total_loss.item()
            epoch_loss += loss_item

            pbar.set_postfix({
                'loss': f'{loss_item:.4f}',
                'avg_loss': f'{epoch_loss/(i+1):.4f}'
            })

            del output_J, output, total_loss

        avg_train_loss = epoch_loss / len(train_dataloader)
        train_losses.append(avg_train_loss)
        print(f"Epoch {epoch} finished. Avg Train Loss: {avg_train_loss:.4f}")

        val_loss = 0.0
        val_psnr = 0.0
        if val_dataloader is not None:
            val_loss, val_psnr = validate_model(model, val_dataloader, loss_1, args)
            print(f"Validation | Loss: {val_loss:.4f} | PSNR: {val_psnr:.2f}")
        val_losses.append(val_loss)
        val_psnrs.append(val_psnr)

        latest_checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_acc': best_acc,
            'val_loss': val_loss
        }
        torch.save(latest_checkpoint, os.path.join(save_path, 'latest_model.pth'))

        if val_psnr > best_acc:
            best_acc = val_psnr
            torch.save(latest_checkpoint, os.path.join(save_path, 'best_model.pth'))
            print(f"Best model PSNR: {val_psnr:.2f}")

        metrics_df = pd.DataFrame({
            'epoch': list(range(1, len(train_losses) + 1)),
            'train_loss': train_losses,
            'val_loss': val_losses,
            'val_psnr': val_psnrs
        })
        metrics_df.to_csv(metrics_path, index=False)

        plt.figure(figsize=(10, 5))
        plt.plot(metrics_df['epoch'], metrics_df['train_loss'], label='Train Loss')
        if val_dataloader is not None:
            plt.plot(metrics_df['epoch'], metrics_df['val_loss'], label='Val Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Loss Curves')
        plt.legend()
        plt.grid(True)
        plt.savefig(os.path.join(save_path, 'loss_curve.png'))
        plt.close()

        if val_dataloader is not None:
            plt.figure(figsize=(10, 5))
            plt.plot(metrics_df['epoch'], metrics_df['val_psnr'], label='Val PSNR')
            plt.xlabel('Epoch')
            plt.ylabel('PSNR')
            plt.title('PSNR Curve')
            plt.legend()
            plt.grid(True)
            plt.savefig(os.path.join(save_path, 'psnr_curve.png'))
            plt.close()

    print("Training completed.")
    print(f"All files saved in: {save_path}")

In [ ]:
class Args:
    train_dir = '/kaggle/input/datasets/adikgupta/bluenet-uieb'   # Must contain: input/, reference-890/, t_prior/, b_prior/
    val_dir = '/kaggle/input/datasets/adikgupta/bluenet-uieb'     # Same structure (can be same as train)
    train_logs = './'                                 # Checkpoints saved to /kaggle/working/save_path/
    seed = 5
    batch_size = 8
    epochs = 200
    lr = 1e-4
    n_layers = 5
    resume = '/kaggle/input/datasets/adikgupta/prevrun/save_path/latest_model.pth'
    num_workers = 4

args = Args()

set_seed(args.seed, torch_set_seed=True)
print('Training started')
training(args)